# 01 — Exploración del Dataset
**VeneCapital | Clasificación de Imágenes Deepfake**

Este notebook realiza:
- Conteo de imágenes por clase
- Visualización de muestras
- Análisis de distribución de dimensiones
- Verificación del balance de clases

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cv2
from pathlib import Path
from collections import Counter
import random
from tqdm import tqdm

from src.config import REAL_DIR, FAKE_DIR, IMG_SIZE

print('✅ Imports exitosos')

In [ ]:
# ── Conteo de imágenes ────────────────────────────────────────────────────
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

real_paths = [p for p in REAL_DIR.iterdir() if p.suffix.lower() in EXTENSIONS]
fake_paths = [p for p in FAKE_DIR.iterdir() if p.suffix.lower() in EXTENSIONS]

print(f'📁 Imágenes REALES: {len(real_paths):,}')
print(f'📁 Imágenes FAKE:   {len(fake_paths):,}')
print(f'📊 Total:           {len(real_paths) + len(fake_paths):,}')
print(f'⚖️  Balance:         {len(real_paths)/(len(real_paths)+len(fake_paths))*100:.1f}% real | {len(fake_paths)/(len(real_paths)+len(fake_paths))*100:.1f}% fake')

In [ ]:
# ── Visualización de muestras ─────────────────────────────────────────────
def show_samples(real_paths, fake_paths, n=5):
    fig, axes = plt.subplots(2, n, figsize=(3*n, 7))
    fig.patch.set_facecolor('#1A1A2E')
    fig.suptitle('Muestras del Dataset', fontsize=16, color='#6C63FF', fontweight='bold')

    for i, path in enumerate(random.sample(real_paths, min(n, len(real_paths)))):
        img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, IMG_SIZE)
        axes[0, i].imshow(img)
        axes[0, i].set_title('REAL', color='#2DD4BF', fontsize=11, fontweight='bold')
        axes[0, i].axis('off')

    for i, path in enumerate(random.sample(fake_paths, min(n, len(fake_paths)))):
        img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, IMG_SIZE)
        axes[1, i].imshow(img)
        axes[1, i].set_title('DEEPFAKE', color='#FF6584', fontsize=11, fontweight='bold')
        axes[1, i].axis('off')

    plt.tight_layout()
    plt.savefig('../outputs/plots/dataset_samples.png', dpi=120, bbox_inches='tight',
                facecolor='#1A1A2E')
    plt.show()

show_samples(real_paths, fake_paths)

In [ ]:
# ── Análisis de dimensiones originales ───────────────────────────────────
print('Analizando dimensiones (puede tardar)...')
widths, heights = [], []
sample = random.sample(real_paths + fake_paths, min(500, len(real_paths) + len(fake_paths)))

for path in tqdm(sample):
    img = cv2.imread(str(path))
    if img is not None:
        h, w = img.shape[:2]
        heights.append(h)
        widths.append(w)

print(f'\n📐 Ancho  — Media: {np.mean(widths):.0f} | Min: {np.min(widths)} | Max: {np.max(widths)}')
print(f'📐 Alto   — Media: {np.mean(heights):.0f} | Min: {np.min(heights)} | Max: {np.max(heights)}')

In [ ]:
# ── Gráfica de balance de clases ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor('#1A1A2E')
ax.set_facecolor('#1A1A2E')

bars = ax.bar(['Real', 'Deepfake'], [len(real_paths), len(fake_paths)],
              color=['#2DD4BF', '#FF6584'], width=0.5, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, [len(real_paths), len(fake_paths)]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{val:,}', ha='center', va='bottom', color='white', fontsize=13, fontweight='bold')

ax.set_title('Balance de Clases en el Dataset', fontsize=15, color='#6C63FF', fontweight='bold')
ax.set_ylabel('Número de Imágenes', color='white')
ax.tick_params(colors='white')
ax.spines['bottom'].set_color('white')
ax.spines['left'].set_color('white')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../outputs/plots/class_balance.png', dpi=120, bbox_inches='tight', facecolor='#1A1A2E')
plt.show()